# XiVLM-Loop: Phase 1 -- Exploratory Data Analysis & Severity Grounding

This notebook performs empirical exploratory data analysis on the industrial PCB defect dataset, computes physical defect severity distributions S(c), and analyzes class balance and false-negative implications.

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from src.severity import compute_severity, CLASS_SEVERITY_TIERS

data_index_path = os.path.join("..", "data", "pcb_defects", "dataset_index.json")
with open(data_index_path, "r", encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
df["severity_score"] = [compute_severity(row["primary_label"], row["area_ratio"]) for _, row in df.iterrows()]
df.head()

## 1. Class Balance & Resolution Statistics

In [ ]:
print("Total Images:", len(df))
print("Resolution:", df["width"].iloc[0], "x", df["height"].iloc[0])
print("Channels: 3 (RGB)")
print("\nClass Breakdown:")
print(df["primary_label"].value_counts())
print("\nSplit Breakdown:")
print(df["split"].value_counts())

## 2. Defect Size & Severity Distribution

Defect severity S(c) is parameterized by industrial risk tier and scaled by defect footprint percentage.

In [ ]:
df.groupby("primary_label")[["area_ratio", "severity_score"]].describe()

## 3. Data Quality & Imbalance Analysis

- **Class Imbalance**: High-severity defects such as `short_circuit` and `pcb_damage` carry high penalty costs in edge electronics manufacturing. Uncalibrated VLMs tend to suffer higher false-negative rates on subtle solder joint anomalies (`dry_joint`).
- **Ambiguous Boundaries**: Incomplete solder wetting creates non-convex polygons where bounding boxes capture high background noise, motivating the multi-modal Faithfulness Guard in Phase 3.